# Bifurcation failure types on 200-track events — amplitudes, eigenvalues, and the 1BQF phase

**Goal.** Use the **classical** solver on **T = 200** events to characterise the
false-positive (FP) failure types: *which segments* go falsely active, *what
amplitude* they reach, and — crucially — *which eigenmode of their cluster* carries
that amplitude (hence at which **QPE phase** $\varphi=\lambda/2(\gamma+\delta)$ the
false weight lives). Then ask the headline question:

> Can we **update the phase of the 1BQF** (move/add a notch) to erase the most
> problematic false types — the way the single notch at $\lambda=\gamma+\delta$
> ($\varphi=0.5$) already erases the **isolated** false segments?

**Background (carried in from the segment-level studies).**
Base segment Hamiltonian $A_0=(\gamma+\delta)I-C$, $b=\delta\mathbf 1$, segment
active iff $x_i>\tau=\delta/(\delta+\gamma)+0.10=0.35$ (at $\gamma=3,\delta=1$,
$s\equiv\gamma+\delta=4$). $C$ = continuation adjacency. The 1BQF applies the
one-bit Hadamard-test filter $f(\lambda)=\cos(\lambda t/2)=\cos(\pi\varphi)$ with
$t=\pi/s$, giving a **single notch (zero) at $\lambda=s$**, i.e. $\varphi=0.5$.
Isolated false segments are a $1\times1$ block with eigenvalue exactly $s$ → they
sit **on** the notch → erased. Coupled clusters (bridges, hubs) do not.

The amplitude atlas (`../Segment_level_studies/07_segment_amplitude_atlas.ipynb`)
and the Run-3 verification both established the FP taxonomy: **false positives come
only from chain≥3 bridges (F3/F4) and hubs (F5)**. Here we measure their spectra
at production scale and test the phase fix.

In [1]:
import sys, os
for _p in ("/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/_shared",
           "/data/bfys/gscriven/LHCb_VeLo_Toy_Model/src"):
    if _p not in sys.path: sys.path.insert(0,_p)
os.environ.setdefault("QTRK_STORE","/data/bfys/gscriven/qtrk_store")
os.environ.setdefault("MPLBACKEND","Agg")
from pathlib import Path
import numpy as np, scipy.sparse as sp, pandas as pd
from scipy.sparse.csgraph import connected_components
import matplotlib.pyplot as plt
import qtrk_pipeline as qp
plt.rcParams.update({"figure.dpi":120,"font.size":11,"axes.grid":True,"grid.alpha":0.3})
OUT=Path("/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Bifurification/outputs")
OUT.mkdir(parents=True,exist_ok=True)

GAMMA,DELTA,EPS=3.0,1.0,0.002
S=GAMMA+DELTA                       # = 4
TAU=qp.threshold_for(GAMMA,DELTA)   # = 0.35
T1=np.pi/S                          # base 1BQF evolution time -> notch at lambda=S
def phi_of(lam): return lam/(2*S)   # QPE phase
print(f"gamma={GAMMA} delta={DELTA}  s=gamma+delta={S}  tau={TAU}  t1=pi/s={T1:.4f}")
print(f"base notch: lambda={S} <-> phi=0.5   (filter f(lambda)=cos(lambda*t1/2)=cos(pi*phi))")

gamma=3.0 delta=1.0  s=gamma+delta=4.0  tau=0.35  t1=pi/s=0.7854
base notch: lambda=4.0 <-> phi=0.5   (filter f(lambda)=cos(lambda*t1/2)=cos(pi*phi))


## 1. Why a notch erases *isolated* false but not *coupled* false — the mechanism

A cluster's block $A_{\rm clu}$ has eigenpairs $(\lambda_k,u_k)$. The classical
solution restricted to the cluster is
$$x = A_{\rm clu}^{-1}b = \sum_k \frac{\beta_k}{\lambda_k}\,u_k,\qquad \beta_k=u_k^\top b,$$
so segment $i$'s amplitude is $x_i=\sum_k c_k\,u_k(i)$ with $c_k=\beta_k/\lambda_k$.
The 1BQF returns the **filtered** solve (the one-bit filter multiplies each mode by
$f(\lambda_k)$, then we renormalise to the classical signal support):
$$x_i^{Q}\;\propto\;\sum_k \beta_k\,f(\lambda_k)\,u_k(i),\qquad f(\lambda)=\cos(\lambda t/2).$$

- **Isolated false** = a $1\times1$ block, single eigenvalue $\lambda=s$. The notch
  $f(s)=\cos(\pi/2)=0$ zeros it outright — a *degenerate* line on the notch with
  **no true segment there**, so it is removed cleanly.
- **Coupled false** (bridges, hubs) = multi-segment blocks whose dominant modes lie
  at $\lambda<s$ — **spread out**, and *overlapping the true-track band*. No single
  notch sits on all of them, and any notch that does hits true tracks too.

We now measure the actual $(\lambda,\text{amplitude})$ of the false positives at
T=200 and see exactly how they sit relative to the true band and the notch.

In [2]:
# --- accumulate false positives over many 200-track events (standard clean config) ---
def cluster_blocks(A_csr, n):
    """connected components of the continuation graph C (off-diagonal support of s*I-A)."""
    Cm=(S*sp.identity(n,format='csr')-A_csr); Cm.setdiag(0); Cm.eliminate_zeros()
    Cm=(abs(Cm)>1e-9).astype(np.int8)
    _,lab=connected_components(Cm,directed=False)
    deg=np.asarray(Cm.sum(1)).ravel()       # continuation degree
    return lab, deg

def solve_event(rep, n_trk=200, ss=1e-4, sr=0.0, phi_max=0.2):
    ev=qp.ensure_event(n_trk=n_trk,rep=rep,sigma_scatt=ss,sigma_res=sr,phi_max=phi_max,hit_ineff=0.0)[0]
    ham=qp.build_hamiltonian(ev,epsilon=EPS,gamma=GAMMA,delta=DELTA); A=ham.A.tocsr(); n=ham.n_segments
    truth=np.asarray(qp.truth_from_event(ev),bool); b=DELTA*np.ones(n)
    sol=np.asarray(sp.linalg.minres(A.tocsc(),b,rtol=1e-9,maxiter=8000)[0])
    lab,deg=cluster_blocks(A,n)
    return A,b,truth,sol,lab,deg,n

N_REPS=30
fp_rows=[]                 # per false-positive segment
clusters=[]                # (w,U,beta,xC,truth_mask,fp_mask,maxdeg) for true+FP clusters -> phase scan
true_dom=[]                # dominant lambda of sampled true-active segments
n_fp_per_event=[]
for rep in range(N_REPS):
    A,b,truth,sol,lab,deg,n=solve_event(rep)
    fa=np.where((~truth)&(sol>TAU))[0]              # false-active segments
    n_fp_per_event.append(len(fa))
    want=set(lab[fa])
    want|=set(lab[np.where(truth&(sol>TAU))[0]][::40])   # a few true clusters for the scan / true band
    for c in want:
        mem=np.where(lab==c)[0]
        if len(mem)>60: continue                    # skip rare giant tangles (keep eigendecomp cheap)
        pos={m:k for k,m in enumerate(mem)}
        sub=A[mem][:,mem].toarray(); w,U=np.linalg.eigh(sub); beta=U.T@b[mem]
        ck=beta/w; maxdeg=int(deg[mem].max())
        tr_m=truth[mem]; fp_m=np.isin(mem,fa)
        clusters.append((w,U,beta,sol[mem],tr_m,fp_m,maxdeg))
        # record per-FP-segment dominant mode + amplitude + type
        for i in np.where(fp_m)[0]:
            contrib=ck*U[i]                          # per-mode contribution to x_i
            kdom=int(np.argmax(np.abs(contrib)))
            ntrk=len(set(int(t) for t in np.atleast_1d(truth[mem]))) # placeholder; real ntrk below
            fp_rows.append(dict(rep=rep, clu_size=len(mem), maxdeg=maxdeg,
                                ftype=("hub" if maxdeg>=3 else "bridge"),
                                lam_dom=float(w[kdom]), phi_dom=float(phi_of(w[kdom])),
                                amp=float(sol[mem][i]), lam_min=float(w.min())))
        if c in set(lab[np.where(truth&(sol>TAU))[0]]):
            for i in np.where(tr_m & (sol[mem]>TAU))[0]:
                contrib=ck*U[i]; kd=int(np.argmax(np.abs(contrib))); true_dom.append(float(w[kd]))
FP=pd.DataFrame(fp_rows); true_dom=np.array(true_dom)
print(f"events: {N_REPS}   total FP segments: {len(FP)}   mean FP/event: {np.mean(n_fp_per_event):.2f}")
print(f"FP type split:  bridge={ (FP.ftype=='bridge').sum() }   hub={ (FP.ftype=='hub').sum() }")
print(f"FP dominant lambda:  min={FP.lam_dom.min():.3f}  median={FP.lam_dom.median():.3f}  max={FP.lam_dom.max():.3f}")
print(f"FP amplitude:        min={FP.amp.min():.3f}  median={FP.amp.median():.3f}  max={FP.amp.max():.3f}")
print(f"TRUE dominant lambda (sampled, n={len(true_dom)}): median={np.median(true_dom):.3f}  (= P4 chain low mode)")

events: 30   total FP segments: 179   mean FP/event: 5.97
FP type split:  bridge=42   hub=137
FP dominant lambda:  min=0.764  median=2.152  max=2.586
FP amplitude:        min=0.357  median=0.392  max=1.500
TRUE dominant lambda (sampled, n=936): median=2.382  (= P4 chain low mode)


## 2. The false-positive atlas at T=200 — type, amplitude, eigenvalue

The true-track low mode sits at a single sharp value
$\lambda_1=(\gamma+\delta)-2\cos(\pi/5)=2.382$ ($\varphi=0.298$). The false positives
fall into the two coupled families:

- **Bridges** (chain≥3 cross-track continuation runs, atlas F3/F4): dominant
  $\lambda$ around the true band and just above it ($\lambda\!\approx\!2.4\text{–}2.6$)
  — they are *graph-isomorphic to true tracks* (the topological degeneracy), so
  they are spectrally indistinguishable from real tracks.
- **Hubs** (high-degree stars, atlas F5): higher continuation degree pushes the
  dominant mode **down** to $\lambda=(\gamma+\delta)-\sqrt{m}$ (e.g. $m=3\to2.27$,
  $m=4\to2.0$, $m=5\to1.76$) — *below* the true band, and carrying a higher
  amplitude (the hub centre).

In [3]:
fig,ax=plt.subplots(1,3,figsize=(16,4.8))
col={"bridge":"#d6604d","hub":"#6a3d9a"}
# (a) type breakdown
vc=FP.ftype.value_counts()
ax[0].bar(vc.index,vc.values,color=[col[k] for k in vc.index],edgecolor="k")
for i,(k,v) in enumerate(vc.items()): ax[0].text(i,v+0.5,f"{v}\n({100*v/len(FP):.0f}%)",ha="center",fontweight="bold")
ax[0].set_title("(a) FP failure type (T=200, classical)",fontweight="bold")
ax[0].set_ylabel("# false-positive segments"); ax[0].set_ylim(0,vc.max()*1.25)
# (b) amplitude vs dominant lambda, colored by type
for k,d in FP.groupby("ftype"):
    ax[1].scatter(d.lam_dom,d.amp,s=28,c=col[k],alpha=0.6,edgecolor="none",label=k)
ax[1].axvline(2.382,color="#1b7837",ls="-",lw=2,label="true low mode λ=2.382")
ax[1].axvline(S,color="k",ls="--",lw=1.5,label=f"notch λ={S} (φ=0.5)")
ax[1].axhline(TAU,color="k",ls=":",lw=1,label=f"τ={TAU}")
ax[1].set_xlabel("dominant eigenvalue λ* of the FP segment")
ax[1].set_ylabel("classical amplitude $x_i$")
ax[1].set_title("(b) FP amplitude vs eigenvalue",fontweight="bold"); ax[1].legend(fontsize=8)
# (c) dominant-lambda distribution: FP vs true
edges=np.arange(0.5,4.01,0.2)
ax[2].hist(true_dom,bins=edges,color="#1b7837",alpha=0.55,density=True,label="true (active)")
ax[2].hist(FP.lam_dom,bins=edges,color="#d6604d",alpha=0.55,density=True,label="false positive")
ax[2].axvline(S,color="k",ls="--",lw=1.5,label=f"notch λ={S}")
ax[2].axvline(2.382,color="#1b7837",ls="-",lw=1)
ax[2].set_xlabel("dominant eigenvalue λ*"); ax[2].set_ylabel("density")
ax[2].set_title("(c) where the false weight lives",fontweight="bold"); ax[2].legend(fontsize=8)
fig.tight_layout()
for e,dp in (("pdf",600),("png",150)): fig.savefig(OUT/f"fp_atlas_T200.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print("saved fp_atlas_T200")

saved fp_atlas_T200


**Read-off.** The notch at $\lambda=s=4$ is far to the right of *all* the coupled
false weight (which lives at $\lambda\lesssim2.6$). The isolated false (not shown —
they are the $1\times1$ blocks pinned exactly at $\lambda=4$) are the only family
the notch touches. The coupled FPs **straddle the true low-mode line at 2.382**:
hubs below it, bridges on/above it. That straddle is the obstruction to a clean
phase fix, which §4 makes quantitative.

## 3. The QPE-phase view — the one-bit filter and where each family sits

Map every eigenvalue to its QPE phase $\varphi=\lambda/2s\in[0,1]$ and overlay the
one-bit filter $f(\varphi)=\cos(\pi\varphi)$. The filter **passes** low phases
($\varphi<0.5$, $f>0$) and has its only in-band zero at $\varphi=0.5$. The
activation-carrying weight of *both* true and coupled-false segments lives at
$\varphi\approx0.25\text{–}0.32$ — squarely in the pass-band — which is why the
single notch cannot reach it.

In [4]:
fig,ax=plt.subplots(1,2,figsize=(14,5))
# (a) filter with the families marked
pp=np.linspace(0.0,1.0,400)
ax[0].plot(pp,np.cos(np.pi*pp),color="#2166ac",lw=2.5,label=r"$f(\varphi)=\cos(\pi\varphi)$")
ax[0].axhline(0,color="k",lw=0.8); ax[0].fill_between(pp,np.cos(np.pi*pp),0,where=(np.cos(np.pi*pp)>0),color="#2166ac",alpha=0.08)
ax[0].axvline(0.5,color="k",ls="--",lw=1.5,label=r"notch $\varphi=0.5$ (isolated false)")
ax[0].axvline(0.298,color="#1b7837",lw=2,label=r"true low mode $\varphi=0.298$")
# coupled-false phase band (dominant)
lo,hi=phi_of(FP.lam_dom.quantile(0.1)),phi_of(FP.lam_dom.quantile(0.9))
ax[0].axvspan(lo,hi,color="#d6604d",alpha=0.18,label=f"coupled-false band φ∈[{lo:.2f},{hi:.2f}]")
ax[0].set_xlabel(r"QPE phase $\varphi=\lambda/2s$"); ax[0].set_ylabel(r"filter $f(\varphi)$")
ax[0].set_title("(a) one-bit filter — only $\\varphi=0.5$ is notched",fontweight="bold")
ax[0].legend(fontsize=8,loc="lower left"); ax[0].set_ylim(-1.05,1.05)
# (b) phase histogram of FP dominant modes by type, with the filter overlaid (twin axis)
ax2=ax[1]
edges=np.linspace(0.1,0.55,24)
ax2.hist(phi_of(FP[FP.ftype=='hub'].lam_dom),bins=edges,color="#6a3d9a",alpha=0.6,label="hub FP")
ax2.hist(phi_of(FP[FP.ftype=='bridge'].lam_dom),bins=edges,color="#d6604d",alpha=0.6,label="bridge FP")
ax2.axvline(0.298,color="#1b7837",lw=2,label="true φ=0.298")
ax2.set_xlabel(r"dominant QPE phase $\varphi$"); ax2.set_ylabel("# FP segments")
axb=ax2.twinx(); axb.plot(pp,np.cos(np.pi*pp),color="#2166ac",lw=2,alpha=0.7); axb.set_ylabel(r"$f(\varphi)$",color="#2166ac"); axb.set_ylim(-1.05,1.05); axb.grid(False)
ax2.set_title("(b) FP dominant phase by type vs the filter",fontweight="bold"); ax2.legend(fontsize=8,loc="upper left")
fig.tight_layout()
for e,dp in (("pdf",600),("png",150)): fig.savefig(OUT/f"phase_filter_map.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print("saved phase_filter_map")

saved phase_filter_map


## 4. Can a phase update erase the coupled false? — the two-notch scan

The single notch must **stay** at $\lambda=s$ or the isolated-false bulk (the vast
majority of all false segments) un-erases. So a phase update means **adding** a
second zero: apply the Hadamard-test filter twice, at $t_1=\pi/s$ and $t_2=\pi/\lambda_2$,
$$f(\lambda)=\cos(\lambda t_1/2)\,\cos(\lambda t_2/2)\quad\Rightarrow\quad\text{zeros at }\lambda=s\ \text{and}\ \lambda=\lambda_2.$$
We scan the second-notch position $\lambda_2$, apply the filter per cluster,
renormalise to the **true** signal support (the physical single global scale of the
1BQF), threshold at $\tau$, and read off the fraction of true-active kept (efficiency)
and the fraction of coupled-FP kept (the thing we want to drive down).

In [5]:
def evaluate(filt):
    """apply mode filter to every collected cluster; rescale to true-active norm; threshold."""
    xCT=[]; xQT=[]; xQF=[]
    for (w,U,beta,xC,tr,fp,md_) in clusters:
        xq=np.abs(U@(beta*filt(w)))
        xCT.append(xC[tr]); xQT.append(xq[tr]); xQF.append(xq[fp])
    xCT=np.concatenate(xCT); xQT=np.concatenate(xQT)
    xQF=np.concatenate(xQF) if any(len(a) for a in xQF) else np.array([])
    mT=xCT>TAU
    scale=np.linalg.norm(xCT[mT])/max(np.linalg.norm(xQT[mT]),1e-12)
    eff=(xQT[mT]*scale>TAU).mean()
    fp_kept=((xQF*scale)>TAU).mean() if xQF.size else 0.0
    return eff,fp_kept,int(mT.sum()),int(xQF.size)

f_base=lambda w: np.cos(w*T1/2)
def f_two(w,l2): return np.cos(w*T1/2)*np.cos(w*(np.pi/l2)/2)

e0,fk0,nT,nF=evaluate(f_base)
L2=np.round(np.arange(1.5,3.21,0.1),2)
eff2=[]; fpk2=[]
for l2 in L2:
    e,fk,_,_=evaluate(lambda w,L=l2: f_two(w,L)); eff2.append(e); fpk2.append(fk)
eff2=np.array(eff2); fpk2=np.array(fpk2)
res=pd.DataFrame({"lambda2":L2,"phi2":(L2/(2*S)).round(3),"true_kept":np.round(eff2,3),"FP_kept":np.round(fpk2,3)})
print(f"BASE 1BQF (single notch λ={S}):  true_kept={e0:.3f}  FP_kept={fk0:.3f}   (nTrue={nT}, nFP={nF})")
display(res)

fig,ax=plt.subplots(1,2,figsize=(14,5))
ax[0].plot(L2,eff2,'o-',color="#1b7837",lw=2,label="true-active kept (efficiency)")
ax[0].plot(L2,fpk2,'s-',color="#d6604d",lw=2,label="coupled-FP kept")
ax[0].axhline(e0,color="#1b7837",ls=":",lw=1.5,label=f"base efficiency {e0:.2f}")
ax[0].axhline(fk0,color="#d6604d",ls=":",lw=1.5,label=f"base FP-kept {fk0:.2f}")
ax[0].axvline(2.382,color="k",ls="-",lw=1,alpha=0.6,label="true low mode 2.382")
ax[0].set_xlabel(r"second notch position $\lambda_2$"); ax[0].set_ylabel("fraction")
ax[0].set_title("(a) 2-notch scan: no λ₂ lowers FP without killing true",fontweight="bold")
ax[0].legend(fontsize=8); ax[0].set_ylim(-0.02,1.05)
# (b) trade-off curve
ax[1].plot(fpk2,eff2,'-',color="#888",lw=1)
sc=ax[1].scatter(fpk2,eff2,c=L2,cmap="viridis",s=55,zorder=3)
ax[1].scatter([fk0],[e0],marker="*",s=380,color="k",zorder=5,edgecolor="w",linewidth=0.8)
ax[1].annotate("base 1BQF\n(single notch)\nPareto-best",(fk0,e0),xytext=(fk0+0.06,e0+0.04),
               fontsize=9,fontweight="bold",ha="left",va="bottom",
               arrowprops=dict(arrowstyle="->",lw=1.2))
ax[1].set_xlabel("coupled-FP kept  (want ←)"); ax[1].set_ylabel("true-active kept  (want ↑)")
ax[1].set_title("(b) efficiency–FP trade-off (colour = λ₂)",fontweight="bold")
plt.colorbar(sc,ax=ax[1],label=r"$\lambda_2$")
fig.tight_layout()
for e,dp in (("pdf",600),("png",150)): fig.savefig(OUT/f"two_notch_scan.{e}",dpi=dp,bbox_inches="tight",facecolor="white")
plt.show(); print("saved two_notch_scan")

BASE 1BQF (single notch λ=4.0):  true_kept=0.535  FP_kept=0.341   (nTrue=936, nFP=179)


,lambda2,phi2,true_kept,FP_kept
0,1.5,0.188,0.880,0.749
1,1.6,0.200,0.882,0.777
2,1.7,0.212,0.505,0.453
3,1.8,0.225,0.511,0.486
4,1.9,0.238,0.498,0.503
5,2.0,0.250,0.444,0.525
6,2.1,0.262,0.394,0.458
7,2.2,0.275,0.085,0.475
8,2.3,0.288,0.157,0.821
9,2.4,0.300,0.160,0.821


saved two_notch_scan


## 5. Conclusion — why the phase trick does **not** generalise, and what does

**The single notch works for the isolated false because they are a degenerate line.**
Isolated false segments are $1\times1$ blocks at exactly $\lambda=s$, with **no true
segment there**. One zero on that line removes the entire family at zero cost.

**The coupled false are not a line — they are a spread that straddles the true band.**
At T=200 the false positives split into:

- **bridges** (chain≥3): dominant $\lambda\approx2.4\text{–}2.6$ — *isomorphic to true
  tracks*, so spectrally **on top of** the true low mode (2.382). No filter that
  passes true tracks can reject them; this is the topological degeneracy, not a
  tunable parameter.
- **hubs**: dominant $\lambda=(\gamma+\delta)-\sqrt m<2.382$ — *just below* the true
  band, by a gap ($\sim0.2$–0.4) far narrower than the one-bit cosine notch is wide.

The two-notch scan (§4) is decisive: there is **no** second-notch position $\lambda_2$
that lowers the coupled-FP fraction without collapsing the true efficiency. Placing
$\lambda_2$ near the hub band ($\approx2.1$) suppresses the true low mode at 2.382
too (the cosine notch is broad), and the true-calibrated renormalisation then
re-inflates the survivors — the FP fraction goes **up**, not down. Placing $\lambda_2$
on the true band annihilates the tracks. **A single extra phase/notch cannot separate
distributions that overlap.**

**What this means for the algorithm.**
1. **Phase tuning is exhausted by the isolated-false notch.** The 1BQF already does
   its one available spectral job; the coupled false cannot be reached by *any*
   choice of evolution time(s), because they share eigenvalues with true tracks.
2. **The fix has to act on the Hamiltonian, not the phase** — exactly the
   **ε-windowed bifurcation term** (`03_epsilon_windowed_bifurcation.ipynb`): it
   adds a *targeted* repulsion between the genuinely competing near-collinear
   bridges inside the acceptance window, drives their classical false-rate to 0 with
   ~2% efficiency cost, stays sparse, and (unlike the dense fork) **keeps the false
   bulk on the notch** so the 1BQF survives. That is the correct lever for the
   bridge family.
3. **Hubs** (the sub-2.382 family) are the residue that neither the phase nor the
   ε-fork removes; their centres carry genuinely high amplitude and need a
   **track-level / occupancy** constraint (one segment per hit), not a segment-level
   spectral cut.

**Bottom line.** The isolated-false erasure was a gift of *degeneracy*: one
eigenvalue, no true neighbours. The problematic coupled false live *inside the true
spectrum*, so the phase has nothing to grab. Removing them is a Hamiltonian-design
problem (the ε-fork term for bridges) and a track-level problem (for hubs) — not a
phase update.